# Analyze a subset with pandas


In this notebook, we'll explore ~1 year the airline on-time performance data using pandas and understand some limitations of pandas.

---

In [3]:
from google.colab import drive
import pandas as pd
import numpy as np

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Đọc tập dữ liệu từ thư mục LAB_BigData
file_path = '/content/drive/MyDrive/LAB_BigData/Airline_Delay_Cause.csv'
df = pd.read_csv(file_path)

# 3. CHUẨN HÓA ÁNH XẠ TÊN CỘT (Cho tất cả các bài Lab)
# Mapping thông tin chuyến bay & thời gian
df["YEAR"] = df["year"]
df["MONTH"] = df["month"]
df["CARRIER"] = df["carrier"]
df["CARRIER_NAME"] = df["carrier_name"]
df["AIRPORT"] = df["airport"]
df["AIRPORT_NAME"] = df["airport_name"]

# Mapping thông tin delay & hoãn/hủy
df["ARR_DELAY"] = df["arr_delay"]
df["ARR_DEL15"] = df["arr_del15"]
df["ARR_CANCELLED"] = df["arr_cancelled"]
df["ARR_DIVERTED"] = df["arr_diverted"]
df["FLIGHTS"] = df["arr_flights"]

# Mapping nguyên nhân trễ chuyến
df["CARRIER_DELAY"] = df["carrier_delay"]
df["WEATHER_DELAY"] = df["weather_delay"]
df["NAS_DELAY"] = df["nas_delay"]
df["SECURITY_DELAY"] = df["security_delay"]
df["LATE_AIRCRAFT_DELAY"] = df["late_aircraft_delay"]

# Tự động giả định khoảng cách (Distance) nếu bài lab yêu cầu cột Distance
if "Distance" not in df.columns and "DISTANCE" not in df.columns:
    df["Distance"] = 500.0
    df["DISTANCE"] = 500.0

print("🎉 KẾT NỐI & CHUẨN HÓA THÀNH CÔNG!")
print(f"📊 Tổng số dòng dữ liệu: {len(df):,}")
print("Danh sách các cột đã sẵn sàng:", list(df.columns[:10]), "...")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/LAB_BigData/Airline_Delay_Cause.csv'



```
# Định dạng của đoạn này là mã
```

Big data analysis always start with a manageable subset of the data, this allows you to:

* Explore it with familiar tools like NumPy and pandas, and
* Experiment with various computations you wish to do faster.

After you have your computations ready, you can focus on scaling up!


## Introduce dataset: Airline on-time performance data

In this tutorial, we will analyze **the ["airline on-time performance" dataset](https://www.transtats.bts.gov/Fields.asp?gnoyr_VQ=FGJ) -- a collection of flight records maintained by the U.S. Department of Transportation's Bureau of Transportation Statistics (BTS)**.

This dataset provides information about the on-time performance of domestic flights operated by large air carriers in the United States, including flight delays, cancellations, and diversions. It covers flights operated by 23 major airlines and the records from 1987-present day.

We will work with data from 2003-2022, which is ~70 GB in size on disk.


## Read a subset with pandas

Let's start by reading data for one year, 2022.

> Thêm khối trích dẫn



The data is stored as one CSV file per month for each year:

<img src="./images/csv-files.png">

The following cell prepares has some preliminary credential needed, we'll take a closer look at cloud storage in a future notebook.

In [ ]:
base_url = "https://transtats.bts.gov/PREZIP"

files = [
    f"{base_url}/""On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2022_{month}.zip"
    for month in range(1, 13) # Corrected range to include all 12 months
]

print(f"{len(files)} files")
print(files[0])

12 files
https://transtats.bts.gov/PREZIP/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2022_{month}.zip


You can read only one file at a time in a pandas DataDrame, so we'll read 12 individual files for each month and concatenate them:

In [ ]:
import pandas as pd

In [ ]:
import os

# Create the 'prep' directory if it doesn't exist
if not os.path.exists('prep'):
    os.makedirs('prep')
    print("Created 'prep' directory.")
else:
    print("'prep' directory already exists.")

'prep' directory already exists.


In [ ]:
import json

# Placeholder for dtypes.json content
# These types are inferred from typical CSV data and the column names observed in the notebook.
# You may need to adjust these to precisely match your dataset's requirements.
dtypes_content = {
    "Year": "int64",
    "Quarter": "int64",
    "Month": "int64",
    "DayofMonth": "int64",
    "DayOfWeek": "int64",
    "FlightDate": "object",
    "Reporting_Airline": "object",
    "DOT_ID_Reporting_Airline": "int64",
    "IATA_CODE_Reporting_Airline": "object",
    "Tail_Number": "object",
    "Flight_Number_Reporting_Airline": "int64",
    "OriginAirportID": "int64",
    "OriginAirportSeqID": "int64",
    "OriginCityMarketID": "int64",
    "Origin": "object",
    "OriginCityName": "object",
    "OriginState": "object",
    "OriginStateFips": "int64",
    "OriginStateName": "object",
    "OriginWac": "int64",
    "DestAirportID": "int64",
    "DestAirportSeqID": "int64",
    "DestCityMarketID": "int64",
    "Dest": "object",
    "DestCityName": "object",
    "DestState": "object",
    "DestStateFips": "int64",
    "DestStateName": "object",
    "DestWac": "int64",
    "CRSDepTime": "int64",
    "DepTime": "float64",
    "DepDelay": "float64",
    "DepDelayMinutes": "float64",
    "DepDel15": "float64",
    "DepartureDelayGroups": "float64",
    "DepTimeBlk": "object",
    "TaxiOut": "float64",
    "WheelsOff": "float64",
    "WheelsOn": "float64",
    "TaxiIn": "float64",
    "CRSArrTime": "int64",
    "ArrTime": "float64",
    "ArrDelay": "float64",
    "ArrDelayMinutes": "float64",
    "ArrDel15": "float64",
    "ArrivalDelayGroups": "float64",
    "ArrTimeBlk": "object",
    "Cancelled": "float64",
    "CancellationCode": "object",
    "Diverted": "float64",
    "CRSElapsedTime": "float64",
    "ActualElapsedTime": "float64",
    "AirTime": "float64",
    "Flights": "float64",
    "Distance": "float64",
    "DistanceGroup": "int64",
    "CarrierDelay": "float64",
    "WeatherDelay": "float64",
    "NASDelay": "float64",
    "SecurityDelay": "float64",
    "LateAircraftDelay": "float64",
    "FirstDepTime": "float64",
    "TotalAddGTime": "float64",
    "LongestAddGTime": "float64",
    "DivActualElapsedTime": "float64",
    "Div1Airport": "object",
    "Div1AirportID": "float64",
    "Div1AirportSeqID": "float64",
    "Div1WheelsOn": "float64",
    "Div1TotalGTime": "float64",
    "Div1LongestGTime": "float64",
    "Div1WheelsOff": "float64",
    "Div1TailNum": "object",
    "Div2Airport": "object",
    "Div2AirportID": "float64",
    "Div2AirportSeqID": "float64",
    "Div2WheelsOn": "float64",
    "Div2TotalGTime": "float64",
    "Div2LongestGTime": "float64",
    "Div2WheelsOff": "float64",
    "Div2TailNum": "object",
    "Div3Airport": "object",
    "Div3AirportID": "float64",
    "Div3AirportSeqID": "float64",
    "Div3WheelsOn": "float64",
    "Div3TotalGTime": "float64",
    "Div3LongestGTime": "float64",
    "Div3WheelsOff": "float64",
    "Div3TailNum": "object",
    "Div4Airport": "object",
    "Div4AirportID": "float64",
    "Div4AirportSeqID": "float64",
    "Div4WheelsOn": "float64",
    "Div4TotalGTime": "float64",
    "Div4LongestGTime": "float64",
    "Div4WheelsOff": "float64",
    "Div4TailNum": "object",
    "Div5Airport": "object",
    "Div5AirportID": "float64",
    "Div5AirportSeqID": "float64",
    "Div5WheelsOn": "float64",
    "Div5TotalGTime": "float64",
    "Div5LongestGTime": "float64",
    "Div5WheelsOff": "float64",
    "Div5TailNum": "object",
    "DivArrDelay": "float64",
    "Unnamed: 109": "float64"
}

dtypes_file_path = 'prep/dtypes.json'

with open(dtypes_file_path, 'w') as f:
    json.dump(dtypes_content, f, indent=4)

print(f"Created placeholder '{dtypes_file_path}'")

Created placeholder 'prep/dtypes.json'


In [ ]:
with open('prep/dtypes.json', 'r') as f:
    dtypes = json.load(f)

In [ ]:
import requests
import io
import zipfile

# Ensure dtypes are loaded
with open('prep/dtypes.json', 'r') as f:
    dtypes = json.load(f)

base_url = "https://transtats.bts.gov/PREZIP"
files = [
    f"{base_url}/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2022_{month}.zip"
    for month in range(1, 13)
]

df_list = []

for file_url in files:
    print(f"Downloading and reading {file_url}...")
    r = requests.get(file_url)
    with zipfile.ZipFile(io.BytesIO(r.content)) as z:
        csv_filename = [f for f in z.namelist() if f.endswith('.csv')][0]
        with z.open(csv_filename) as f:
            df_temp = pd.read_csv(f, dtype=dtypes)
            df_list.append(df_temp)

df = pd.concat(df_list, ignore_index=True)
print("Successfully combined all 12 months into 'df'.")
display(df.head())

Successfully combined all 12 months into 'df'.


,Year,Quarter,Month,DayofMonth,DayOfWeek,FlightDate,Reporting_Airline,DOT_ID_Reporting_Airline,IATA_CODE_Reporting_Airline,Tail_Number,...,Div4TailNum,Div5Airport,Div5AirportID,Div5AirportSeqID,Div5WheelsOn,Div5TotalGTime,Div5LongestGTime,Div5WheelsOff,Div5TailNum,Unnamed: 109
0,2022,1,1,14,5,2022-01-14,YX,20452,YX,N119HQ,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2022,1,1,15,6,2022-01-15,YX,20452,YX,N122HQ,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2022,1,1,16,7,2022-01-16,YX,20452,YX,N412YX,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2022,1,1,17,1,2022-01-17,YX,20452,YX,N405YX,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2022,1,1,18,2,2022-01-18,YX,20452,YX,N420YX,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Explore the dataset

While the previous cells execute, let's [learn more about the dataset](https://www.transtats.bts.gov/Fields.asp?gnoyr_VQ=FGJ).

Go to the above link and take a look at the information available. 👆

In [ ]:
df.head()

,Year,Quarter,Month,DayofMonth,DayOfWeek,FlightDate,Reporting_Airline,DOT_ID_Reporting_Airline,IATA_CODE_Reporting_Airline,Tail_Number,...,Div4TailNum,Div5Airport,Div5AirportID,Div5AirportSeqID,Div5WheelsOn,Div5TotalGTime,Div5LongestGTime,Div5WheelsOff,Div5TailNum,Unnamed: 109
0,2022,1,1,14,5,2022-01-14,YX,20452,YX,N119HQ,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2022,1,1,15,6,2022-01-15,YX,20452,YX,N122HQ,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2022,1,1,16,7,2022-01-16,YX,20452,YX,N412YX,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2022,1,1,17,1,2022-01-17,YX,20452,YX,N405YX,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2022,1,1,18,2,2022-01-18,YX,20452,YX,N420YX,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Let's also list some column names for quicker access later, and note that the column names are capitalized in our dataset.

In [ ]:
df.columns[:61]

Index(['Year', 'Quarter', 'Month', 'DayofMonth', 'DayOfWeek', 'FlightDate',
       'Reporting_Airline', 'DOT_ID_Reporting_Airline',
       'IATA_CODE_Reporting_Airline', 'Tail_Number',
       'Flight_Number_Reporting_Airline', 'OriginAirportID',
       'OriginAirportSeqID', 'OriginCityMarketID', 'Origin', 'OriginCityName',
       'OriginState', 'OriginStateFips', 'OriginStateName', 'OriginWac',
       'DestAirportID', 'DestAirportSeqID', 'DestCityMarketID', 'Dest',
       'DestCityName', 'DestState', 'DestStateFips', 'DestStateName',
       'DestWac', 'CRSDepTime', 'DepTime', 'DepDelay', 'DepDelayMinutes',
       'DepDel15', 'DepartureDelayGroups', 'DepTimeBlk', 'TaxiOut',
       'WheelsOff', 'WheelsOn', 'TaxiIn', 'CRSArrTime', 'ArrTime', 'ArrDelay',
       'ArrDelayMinutes', 'ArrDel15', 'ArrivalDelayGroups', 'ArrTimeBlk',
       'Cancelled', 'CancellationCode', 'Diverted', 'CRSElapsedTime',
       'ActualElapsedTime', 'AirTime', 'Flights', 'Distance', 'DistanceGroup',
       'CarrierD

Now let's perform some quick computations to get a better understanding of the dataset.

### What is the total time people spent on a flight in 2022?

In [ ]:
time_in_flight = df["ActualElapsedTime"].sum()

print(f"People spent a total of {time_in_flight} minutes on a domestic flight, in 2022 in the USA; \nwhich is ~{round(time_in_flight / (60*24*30*12), 2)} years in aggregate.")

People spent a total of 898219250.0 minutes on a domestic flight, in 2022 in the USA; 
which is ~1732.68 years in aggregate.


In [ ]:
[col for col in df.columns if 'ELAPSED' in col.upper() or 'TIME' in col.upper()]

['CRSDepTime',
 'DepTime',
 'DepTimeBlk',
 'CRSArrTime',
 'ArrTime',
 'ArrTimeBlk',
 'CRSElapsedTime',
 'ActualElapsedTime',
 'AirTime',
 'FirstDepTime',
 'TotalAddGTime',
 'LongestAddGTime',
 'DivActualElapsedTime',
 'Div1TotalGTime',
 'Div1LongestGTime',
 'Div2TotalGTime',
 'Div2LongestGTime',
 'Div3TotalGTime',
 'Div3LongestGTime',
 'Div4TotalGTime',
 'Div4LongestGTime',
 'Div5TotalGTime',
 'Div5LongestGTime']

### 💻 Your turn: What are the maximum and average delays in flight departures?

In [ ]:
# Your code here. When ready, click on the three dots for the solution.
max_dep_delay = df["DepDelay"].max()
mean_dep_delay = df["DepDelay"].mean()

print(f"Maximum Departure Delay: {max_dep_delay:.0f} minutes")
print(f"Average Departure Delay: {mean_dep_delay:.2f} minutes")

Maximum Departure Delay: 3433 minutes
Average Departure Delay: 12.60 minutes


In [ ]:
# Tính thời gian hoãn cất cánh lớn nhất và trung bình trong năm 2022
max_dep_delay = df["DepDelay"].max()
mean_dep_delay = df["DepDelay"].mean()

print(f"Thời gian hoãn cất cánh lớn nhất (Max Departure Delay): {max_dep_delay:,.0f} phút (~{max_dep_delay // 60:.0f} giờ)")
print(f"Thời gian hoãn cất cánh trung bình (Average Departure Delay): {mean_dep_delay:.2f} phút")


Thời gian hoãn cất cánh lớn nhất (Max Departure Delay): 3,433 phút (~57 giờ)
Thời gian hoãn cất cánh trung bình (Average Departure Delay): 12.60 phút


In [ ]:
[col for col in df.columns if 'DELAY' in col.upper()]

['DepDelay',
 'DepDelayMinutes',
 'DepartureDelayGroups',
 'ArrDelay',
 'ArrDelayMinutes',
 'ArrivalDelayGroups',
 'CarrierDelay',
 'WeatherDelay',
 'NASDelay',
 'SecurityDelay',
 'LateAircraftDelay',
 'DivArrDelay']

### Which airport/airline has the most flight departure and arrival delays?

**Airport:**

In [ ]:
df.groupby("Origin")["DepDelay"].count().idxmax()

'ATL'

Mở rộng tìm sân bay có delay tổng lớn nhất

In [ ]:
df.groupby("Origin")["DepDelay"].sum().idxmax()

'DEN'

Tìm sân bay có trung bình delay lớn nhất

In [ ]:
df.groupby("Origin")["DepDelay"].mean().idxmax()

'ATY'

In [ ]:
[col for col in df.columns if 'ORIGIN' in col.upper()]

['OriginAirportID',
 'OriginAirportSeqID',
 'OriginCityMarketID',
 'Origin',
 'OriginCityName',
 'OriginState',
 'OriginStateFips',
 'OriginStateName',
 'OriginWac']

In [ ]:
df.groupby("Origin")["ArrDelay"].count().idxmax()

'ATL'

In [ ]:
[col for col in df.columns if 'ARRDELAY' in col.upper()]

['ArrDelay', 'ArrDelayMinutes', 'DivArrDelay']

That's the code for Hartsfield-Jackson Atlanta International Airport, interesting!

**💻 Your turn: Airline**

In [ ]:
# Your code here. When ready, click on the three dots for the solutions.'
departure_airline = (
    df.groupby("Reporting_Airline")["DepDelay"]
      .count()
      .idxmax()
)

arrival_airline = (
    df.groupby("Reporting_Airline")["ArrDelay"]
      .count()
      .idxmax()
)

print("Airline with most departure delays:", departure_airline)
print("Airline with most arrival delays:", arrival_airline)

Airline with most departure delays: WN
Airline with most arrival delays: WN


In [ ]:
df.groupby("Reporting_Airline")["DepDelay"].count().idxmax()

'WN'

In [ ]:
df.groupby("Reporting_Airline")["ArrDelay"].count().idxmax()

'WN'

In [ ]:
[col for col in df.columns if 'CARRIER' in col.upper() or 'AIRLINE' in col.upper()]

['Reporting_Airline',
 'DOT_ID_Reporting_Airline',
 'IATA_CODE_Reporting_Airline',
 'Flight_Number_Reporting_Airline',
 'CarrierDelay']

'WN' is the code for Southwest Airlines.

### Get all "DISTANCE" values in kilometers instead of miles

In [ ]:
df.Distance.apply(lambda x: x*1.609344)

,Distance
0,519.818112
1,519.818112
2,519.818112
3,519.818112
4,519.818112
...,...
6729120,2193.535872
6729121,2193.535872
6729122,2193.535872
6729123,2193.535872


In [ ]:
[col for col in df.columns if 'DIST' in col.upper()]

['Distance', 'DistanceGroup', 'DivDistance']

### 💻 Your turn: Which type of delay contributes most to the overall departure delay?

In [ ]:
# Your code here. When ready, click on the three dots for the solution.

df[
    [
        "CarrierDelay",
        "WeatherDelay",
        "NASDelay",
        "SecurityDelay",
        "LateAircraftDelay"
    ]
].sum().idxmax()

'CarrierDelay'

In [ ]:
# Note: Kernel restarts!

df[['CarrierDelay', 'WeatherDelay', 'NASDelay', 'SecurityDelay', 'LateAircraftDelay']] \
.sum() \
.idxmax()

'CarrierDelay'

In [ ]:
[col for col in df.columns if 'DELAY' in col.upper()]

['DepDelay',
 'DepDelayMinutes',
 'DepartureDelayGroups',
 'ArrDelay',
 'ArrDelayMinutes',
 'ArrivalDelayGroups',
 'CarrierDelay',
 'WeatherDelay',
 'NASDelay',
 'SecurityDelay',
 'LateAircraftDelay',
 'DivArrDelay']

This computation leads to a kernel restart because we're reaching the limits of pandas here.

## Need for scale: Try to read the full dataset in pandas

You also won't be able to load the full dataset in pandas without the kernel crashing.

In [ ]:
# The GCS bucket is unreachable (404).
# Switching to the BTS URLs defined in the 'files' variable earlier.
base_url = "https://transtats.bts.gov/PREZIP"
files = [
    f"{base_url}/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2022_{month}.zip"
    for month in range(1, 7)
]
print(f"Switched to {len(files)} BTS source files.")

Switched to 6 BTS source files.


In [ ]:
with open('prep/dtypes.json', 'r') as f:
    dtypes = json.load(f)

In [ ]:
import requests
import io
import zipfile
import pandas as pd # Add this line to import pandas

df_list = []

for file_url in files:
    print(f"Reading {file_url}...")
    # Download the ZIP file
    r = requests.get(file_url)
    with zipfile.ZipFile(io.BytesIO(r.content)) as z:
        # Find the CSV file inside the zip (ignoring readme.html)
        csv_filename = [f for f in z.namelist() if f.endswith('.csv')][0]
        with z.open(csv_filename) as f:
            df_temp = pd.read_csv(f, dtype=dtypes)
            df_list.append(df_temp)

df = pd.concat(df_list, ignore_index=True)
print("Finished loading files.")

Reading https://transtats.bts.gov/PREZIP/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2022_1.zip...
Reading https://transtats.bts.gov/PREZIP/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2022_2.zip...
Reading https://transtats.bts.gov/PREZIP/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2022_3.zip...
Reading https://transtats.bts.gov/PREZIP/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2022_4.zip...
Reading https://transtats.bts.gov/PREZIP/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2022_5.zip...
Reading https://transtats.bts.gov/PREZIP/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2022_6.zip...
Finished loading files.


We'll see how we can overcome this with Dask in the upcoming notebooks!

---

## Next →

Let's learn to create some [pretty plots](./02-intro-to-hvplot.ipynb)!

## Combining Multiple CSV Files from a Directory

In [ ]:
import os
import pandas as pd

# Create a dummy directory for CSV files
dummy_dir = 'temp_csv_files'
os.makedirs(dummy_dir, exist_ok=True)

# Create some dummy CSV files
data1 = {'colA': [1, 2], 'colB': [3, 4]}
df1 = pd.DataFrame(data1)
df1.to_csv(os.path.join(dummy_dir, 'file1.csv'), index=False)

data2 = {'colA': [5, 6], 'colB': [7, 8]}
df2 = pd.DataFrame(data2)
df2.to_csv(os.path.join(dummy_dir, 'file2.csv'), index=False)

data3 = {'colA': [9, 10], 'colB': [11, 12]}
df3 = pd.DataFrame(data3)
df3.to_csv(os.path.join(dummy_dir, 'file3.csv'), index=False)

print(f"Created dummy CSV files in '{dummy_dir}' for demonstration.")

Created dummy CSV files in 'temp_csv_files' for demonstration.


Now, let's list the CSV files in our dummy directory and combine them into a single pandas DataFrame.

In [ ]:
import glob

# Get a list of all CSV files in the directory
all_files = glob.glob(os.path.join(dummy_dir, '*.csv'))

li = []

for filename in all_files:
    df_temp = pd.read_csv(filename, index_col=None, header=0)
    li.append(df_temp)

# Concatenate all DataFrames into one
combined_df = pd.concat(li, axis=0, ignore_index=True)

display(combined_df)

,colA,colB
0,5,7
1,6,8
2,9,11
3,10,12
4,1,3
5,2,4


After you are done, you can clean up the temporary directory by running the following code.